# 11 · Data Mapping & Normalization (Rule-based, no statistics)

<a id="sec-overview"></a>

**Goal:** Deterministic, target-agnostic cleaning to standardize categorical values, fix obvious numeric errors, and prepare inputs for imputation/encoding.

**Inputs:** `data/raw/train.csv`, `data/raw/test.csv`  
**Outputs:** `data/processed/11_train_clean.csv`, `data/processed/11_test_clean.csv`, `reports/11_clean_report.json`

In this notebook we prepare the Cars4U dataset for modeling by standardizing categorical values, fixing data-quality issues, and exporting clean artifacts for downstream experimentation.

**Key objectives**

- Harmonize inherited labels so train and test share the same categorical vocabulary.
- Cast numeric-like columns to reliable dtypes while preserving the signal of missingness.
- Flag or remove implausible measurements to minimise leakage into later modeling steps.
- Persist processed CSV outputs together with lightweight audits for traceability.


<a id="top"></a>
## Table of Contents

1. [Project Overview](#sec-overview)
2. [Setup & Imports](#sec-setup)
3. [Load Data & Initial Audit](#sec-load)
4. [Type Casting](#sec-type-casting)
5. [Duplicate Removal](#sec-duplicates)
6. [Categorical Normalization](#sec-categorical)
   - [Transmission Mapping](#sec-transmission)
   - [Fuel Type Mapping](#sec-fuel)
   - [Brand Normalization](#sec-brand)
   - [Model Normalization](#sec-model)
   - [Infer Missing Brands](#sec-brand-from-model)
7. [Outlier & Anomaly Handling](#sec-outliers)
   - [Year Before 2000](#sec-outliers-year)
   - [Engine Size Limits](#sec-outliers-engine)
   - [Year Rounding Errors](#sec-outliers-rounding)
   - [Negative Tax Values](#sec-outliers-tax)
   - [Negative Mileage Values](#sec-outliers-mileage)
   - [MPG vs Fuel Type](#sec-outliers-mpg)
   - [Previous Owners](#sec-outliers-owners)
   - [Future Model Years](#sec-outliers-future-year)
   - [Paint Quality Range](#sec-outliers-paint)
   - [Unknown Transmission](#sec-outliers-unknown-transmission)
8. [Export Processed Data](#sec-export)
9. [Appendix: Missing Value Summary](#sec-appendix)

[Back to top](#top)


<a id="sec-setup"></a>
## 1. Setup & Imports

Gather shared dependencies and configure pandas so exploratory outputs remain readable.


In [1]:
# Core libraries, configuration, and reproducibility helpers
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re
import numpy as np
import pandas as pd

# Configure pandas so exploratory outputs remain legible in the notebook
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


<a id="sec-load"></a>
## 2. Load Data & Initial Audit

Load the Kaggle competition files and take a quick look at their structure before cleaning.


Read the train/test CSV exports into pandas DataFrames, confirm their shapes, and surface early warning signs such as missing values.


In [2]:
# Define canonical data paths and read the raw competition splits
data_dir = "../data/"

train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

# Quick shape check and preview to validate the load
print("Loaded shape:", train.shape)
display(train.head(3))


Loaded shape: (75973, 14)


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


In [3]:
# Summarise missing values for the training columns
nan_summary = train.isna().sum().reset_index()
nan_summary.columns = ['Column', 'NaN Count']
nan_summary = nan_summary[nan_summary['NaN Count'] > 0]
nan_summary = nan_summary.sort_values(by='NaN Count', ascending=False)
nan_summary


,Column,NaN Count
9,mpg,7926
8,tax,7904
12,previousOwners,1550
13,hasDamage,1548
11,paintQuality%,1524
5,transmission,1522
1,Brand,1521
2,model,1517
10,engineSize,1516
7,fuelType,1511


<a id="sec-type-casting"></a>
## 3. Type Casting

Standardise numeric-like columns so downstream computations operate on consistent dtypes.


Convert price-related and usage-related fields to floats or nullable integers while keeping track of how many values were coerced to `NaN` along the way.


In [4]:
def to_int_series(s):
    """Coerce a series to pandas nullable integers while preserving missing values."""
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")

def to_float_series(s):
    """Coerce a series to float, leaving non-parsable entries as NaN."""
    return pd.to_numeric(s, errors="coerce").astype(float)

# Track per-column missing counts during dtype enforcement
nan_report = {}

num_cols_suggest = ["price","mileage","engineSize","mpg","tax","year","previousOwners"]
for col in num_cols_suggest:
    if col in train.columns:
        before = train[col].isna().sum()
        if col in ["year", "previousOwners"]:
            train[col] = to_int_series(train[col])
        else:
            train[col] = to_float_series(train[col])


In [5]:
# Verify the resulting pandas dtypes after normalisation
train.dtypes


carID               int64
Brand              object
model              object
year                Int64
price             float64
transmission       object
mileage           float64
fuelType           object
tax               float64
mpg               float64
engineSize        float64
paintQuality%     float64
previousOwners      Int64
hasDamage         float64
dtype: object

<a id="sec-duplicates"></a>
## 4. Duplicate Removal

Ensure each `carID` appears only once per split to avoid leaking duplicate vehicles into the training set.


Reinstate `carID` as a column if it exists in the index, scan for duplicates, and display any offending rows for manual review.


In [6]:
# Align identifier column and surface duplicates in the training set
if "carID" not in train.columns and train.index.name == "carID":
    train = train.reset_index()

# check for duplicate carIDs
dup_mask = train.duplicated(subset=["carID"], keep=False)

# Extract and sort all rows that have duplicate 'carID' values
dup_rows = train.loc[dup_mask].sort_values("carID")

# Count the number of duplicated carID
n_dups = train.duplicated(subset=["carID"]).sum()
print(f"[TRAIN] duplicate carID count: {n_dups}")

if n_dups > 0:
    print("carID values with duplicates:")
    print(dup_rows["carID"].unique())
    print("Rows with duplicated carID:")
    display(dup_rows)
else:
    print("No duplicate carID found.")


[TRAIN] duplicate carID count: 0
No duplicate carID found.


In [7]:
# Mirror the duplicate check for the test split
if "carID" not in test.columns and test.index.name == "carID":
    test = test.reset_index()

dup_mask_test = test.duplicated(subset=["carID"], keep=False)
dup_rows_test = test.loc[dup_mask_test].sort_values("carID")
n_dups_test = test.duplicated(subset=["carID"]).sum()
print(f"[TEST] duplicate carID count: {n_dups_test}")

if n_dups_test > 0:
    print("carID values with duplicates:")
    print(dup_rows_test["carID"].unique())
    print("Rows with duplicated carID:")
    display(dup_rows_test)
else:
    print("No duplicate carID found in test.")


[TEST] duplicate carID count: 0
No duplicate carID found in test.


<a id="sec-categorical"></a>
## 5. Categorical Normalization

Standardise text fields across both splits so shared mapping dictionaries can be applied reliably.


In [8]:
# Normalise all string columns to lowercase before applying curated mappings
for c in train.select_dtypes(include="string").columns:
    train[c] = train[c].str.lower()
for c in test.select_dtypes(include="string").columns:
    test[c] = test[c].str.lower()


<a id="sec-transmission"></a>
### 5.1 Transmission Mapping

Clean raw transmission labels (punctuation, spacing) and map them to a canonical set defined in `mapping/transmission_mapping.json`.


In [9]:
# Helper functions to normalise and map transmission labels consistently
def normalize_transmission(value: str):
    """Lowercase, strip punctuation, and collapse whitespace for mapping keys."""
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().lower()
    value = re.sub(r"[.,_]", " ", value)
    value = " ".join(value.split())
    return value

def apply_transmission_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/transmission_mapping.json",
    col: str = "transmission",
) -> pd.DataFrame:
    """Apply the curated transmission mapping and report unmapped variants."""
    with open(mapping_path, "r", encoding="utf-8") as f:
        raw_map = json.load(f)

    trans_canon = {normalize_transmission(k): v for k, v in raw_map.items()}

    uniq_before = df[col].nunique(dropna=True)
    print(f"[DEBUG] {col}: unique values BEFORE mapping: {uniq_before}")

    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_transmission)
    df[col] = df[norm_col].map(trans_canon)

    uniq_after = df[col].nunique(dropna=True)
    print(f"[DEBUG] {col}: unique values AFTER mapping: {uniq_after}")
    print(f"[DEBUG] {col}: sample AFTER: {df[col].dropna().unique()[:15]}")

    unmapped = df.loc[df[col].isna(), norm_col].dropna().unique()
    if len(unmapped) > 0:
        print(f"[WARN] Unmapped {col} values:")
        for v in unmapped:
            print("  -", repr(v))
    else:
        print(f"[INFO] All {col} values were mapped.")

    df.drop(columns=[norm_col], inplace=True)
    return df

train = apply_transmission_mapping(train)
test = apply_transmission_mapping(test)


[DEBUG] transmission: unique values BEFORE mapping: 40
[DEBUG] transmission: unique values AFTER mapping: 5
[DEBUG] transmission: sample AFTER: ['Semi-Auto' 'Manual' 'Automatic' 'Unknown' 'Other']
[INFO] All transmission values were mapped.
[DEBUG] transmission: unique values BEFORE mapping: 38
[DEBUG] transmission: unique values AFTER mapping: 5
[DEBUG] transmission: sample AFTER: ['Automatic' 'Semi-Auto' 'Manual' 'Unknown' 'Other']
[INFO] All transmission values were mapped.


<a id="sec-fuel"></a>
### 5.2 Fuel Type Mapping

Repeat the normalisation process for `fuelType` so hybrids, EVs, and conventional fuels share the same spelling across splits.


In [10]:
# Normalise and map fuel type labels to the canonical set
def normalize_fuel(value: str):
    """Lowercase fuel strings and unify punctuation prior to mapping."""
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().lower()
    value = re.sub(r"[.,-_]", " ", value)
    value = " ".join(value.split())
    return value

def apply_fuel_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/fueltype_mapping.json",
    col: str = "fuelType",
) -> pd.DataFrame:
    """Apply the curated fuel-type mapping and surface unmapped variants."""
    with open(mapping_path, "r", encoding="utf-8") as f:
        raw_fuel_map = json.load(f)

    fuel_canon = {normalize_fuel(k): v for k, v in raw_fuel_map.items()}

    uniq_before = df[col].nunique(dropna=True)
    print(f"[DEBUG] {col}: unique values BEFORE mapping: {uniq_before}")

    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_fuel)
    df[col] = df[norm_col].map(fuel_canon)

    print(f"[DEBUG] {col}: unique AFTER mapping: {df[col].dropna().unique()[:20]}")

    unmapped = df.loc[df[col].isna(), norm_col].dropna().unique()
    if len(unmapped) > 0:
        print(f"[WARN] Unmapped {col} values (consider updating the JSON):")
        for v in unmapped:
            print("  -", repr(v))
    else:
        print(f"[INFO] All {col} values were mapped.")

    df.drop(columns=[norm_col], inplace=True)
    return df

train = apply_fuel_mapping(train)
test = apply_fuel_mapping(test)


[DEBUG] fuelType: unique values BEFORE mapping: 34
[DEBUG] fuelType: unique AFTER mapping: ['Petrol' 'Diesel' 'Hybrid' 'Other' 'Electric']
[INFO] All fuelType values were mapped.
[DEBUG] fuelType: unique values BEFORE mapping: 29
[DEBUG] fuelType: unique AFTER mapping: ['Petrol' 'Diesel' 'Hybrid' 'Other' 'Electric']
[INFO] All fuelType values were mapped.


<a id="sec-brand"></a>
### 5.3 Brand Normalization

Standardise manufacturer names with curated mappings to consolidate typos and aliasing.


We have typos in our brand names. We create a mapping dictionary to correct them and log any values that still need manual alignment.


In [11]:
# Normalise brand names prior to mapping so lookups are deterministic
def normalize_brand(value: str):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().lower()
    value = re.sub(r"[.,-_]", " ", value)
    value = " ".join(value.split())
    return value

# Clean and standardise brand names using curated dictionaries
def apply_brand_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/brandname_mapping.json",
    col: str = "Brand",
) -> pd.DataFrame:
    if col not in df.columns:
        print(f"[INFO] Column '{col}' not found.")
        return df

    with open(mapping_path, "r", encoding="utf-8") as f:
        raw_brand_map = json.load(f)

    brand_canon = {normalize_brand(k): v for k, v in raw_brand_map.items()}

    print(f"[INFO] {col} BEFORE cleaning:")
    print(df[col].dropna().unique()[:30])

    was_nan_before = df[col].isna()
    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_brand)

    mapped = df[norm_col].map(brand_canon)
    clean_col = f"{col}_clean"
    df[clean_col] = df[col].where(mapped.isna(), mapped)

    new_nans_mask = df[clean_col].isna() & (~was_nan_before)
    new_nans_count = new_nans_mask.sum()

    unmapped = (
        df.loc[mapped.isna() & (~was_nan_before) & df[norm_col].notna(), norm_col]
          .dropna()
          .unique()
    )

    df[col] = df[clean_col]
    df.drop(columns=[norm_col, clean_col], inplace=True)

    print(f"[INFO] {col} AFTER cleaning:")
    print(df[col].dropna().unique()[:30])

    if new_nans_count > 0:
        print(f"[WARN] {new_nans_count} rows became NaN due to missing mapping.")
    if len(unmapped) > 0:
        print("[WARN] Unmapped brand values (add to JSON):")
        for v in unmapped:
            print("  -", repr(v))

    return df

train = apply_brand_mapping(train, "../mapping/brandname_mapping.json", col="Brand")
test = apply_brand_mapping(test,  "../mapping/brandname_mapping.json", col="Brand")


[INFO] Brand BEFORE cleaning:
['VW' 'Toyota' 'Audi' 'Ford' 'BMW' 'Skoda' 'Opel' 'Mercedes' 'FOR'
 'mercedes' 'Hyundai' 'w' 'ord' 'MW' 'bmw' 'yundai' 'BM' 'Toyot' 'udi'
 'Ope' 'AUDI' 'V' 'opel' 'pel' 'For' 'pe' 'Mercede' 'audi' 'MERCEDES'
 'OPEL']
[INFO] Brand AFTER cleaning:
['Volkswagen' 'Toyota' 'Audi' 'Ford' 'BMW' 'Škoda' 'Opel' 'Mercedes-Benz'
 'Hyundai']
[INFO] Brand BEFORE cleaning:
['Hyundai' 'VW' 'BMW' 'Opel' 'Ford' 'Mercedes' 'Skoda' 'Toyot' 'Toyota'
 'Audi' 'For' 'Ope' 'toyota' 'vw' 'hyundai' 'MW' 'SKODA' 'ord' 'udi' 'bmw'
 'V' 'BM' 'HYUNDAI' 'OPEL' 'mercedes' 'audi' 'Mercede' 'pel' 'opel' 'FORD']
[INFO] Brand AFTER cleaning:
['Hyundai' 'Volkswagen' 'BMW' 'Opel' 'Ford' 'Mercedes-Benz' 'Škoda'
 'Toyota' 'Audi']


<a id="sec-model"></a>
### 5.4 Model Normalization

Like brands, model names carry typos and spacing differences. We inspect distinct entries and clean them with curated aliases and regex rules.


In [12]:
# Inspect distinct model values to understand the space before mapping
unique_models = train["model"].dropna().unique().tolist()
print("Unique Model in 'model':", unique_models)
print(f"Number of unique models: {len(unique_models)}")


Unique Model in 'model': [' Golf', ' Yaris', ' Q2', ' FIESTA', ' 2 Series', '3 Series', ' A3', ' Octavia', ' Passat', ' Focus', ' Insignia', ' A Clas', ' Q3', ' Fabia', ' A Class', ' Ka+', ' 3 Series', ' GLC Class', ' I30', ' C Class', ' Polo', ' E Class', ' C Clas', ' Q5', ' Up', ' Fiesta', ' C-HR', ' Mokka X', ' Corsa', ' Astra', ' TT', ' 5 Series', ' Aygo', ' 4 Series', ' SLK', ' Viva', ' T-Roc', 'Focus', ' EcoSport', ' Tucson', ' EcoSpor', ' X-CLASS', ' CL Class', ' IX20', ' i20', ' Rapid', ' a1', ' Auris', ' sharan', ' I20', ' Adam', ' X3', ' A8', ' GLS Class', ' B-MAX', ' A4', ' Kona', ' I10', ' A1', ' Mokka', ' fiesta', ' S-MAX', ' X2', ' Crossland X', ' Tiguan', ' A5', ' GLE Class', ' C CLASS', ' mokka x', ' Zafira', ' Ioniq', ' A6', ' Mondeo', ' Yeti Outdoor', ' X1', 'POLO', ' INSIGNIA', ' Scala', ' S Class', ' 1 Series', ' Kamiq', ' Kuga', ' Tourneo Connect', ' Q7', ' GLA Class', ' Arteon', ' polo', ' SL CLASS', 'Tucson', ' Santa Fe', ' Grandland X', ' I800', ' ASTRA', ' RAV4

In [13]:
import json, re
import pandas as pd

# Normalise model names before mapping to collapse spelling variations
def norm_model(s):
    if pd.isna(s):
        return pd.NA
    s = str(s).strip().lower()
    s = re.sub(r"[.,\-_ ]+", "", s)
    return s

# Clean and standardise model names using curated dictionaries
def apply_model_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/modelname_mapping.json",
    col: str = "model",
) -> pd.DataFrame:
    with open(mapping_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    raw_aliases = cfg["aliases"]
    regex_rules = cfg.get("regex_rules", [])

    aliases = {norm_model(k): v for k, v in raw_aliases.items()}

    def apply_regex_first(val: str) -> str:
        for rule in regex_rules:
            pat = rule["pattern"]
            repl = rule["replace"]
            match = re.fullmatch(pat, val)
            if match:
                return match.expand(repl)
        return val

    if col not in df.columns:
        print(f"[INFO] column '{col}' not in df, skipping.")
        return df

    print(f"[INFO] {col} BEFORE mapping: {df[col].nunique(dropna=True)} unique")

    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(norm_model)

    def map_model(norm_val):
        if pd.isna(norm_val):
            return pd.NA
        norm_val = apply_regex_first(norm_val)
        return aliases.get(norm_val, pd.NA)

    df[f"{col}_mapped"] = df[norm_col].apply(map_model)
    df[col] = df[f"{col}_mapped"]

    mask_unmapped = df[col].isna() & df[norm_col].notna()
    df.loc[mask_unmapped, col] = pd.NA

    df.drop(columns=[norm_col, f"{col}_mapped"], inplace=True)

    print(f"[INFO] {col} AFTER mapping: {df[col].nunique(dropna=True)} unique")
    return df

train = apply_model_mapping(train, "../mapping/modelname_mapping.json", col="model")
test = apply_model_mapping(test,  "../mapping/modelname_mapping.json", col="model")


[INFO] model BEFORE mapping: 735 unique
[INFO] model AFTER mapping: 185 unique
[INFO] model BEFORE mapping: 593 unique
[INFO] model AFTER mapping: 177 unique


<a id="sec-brand-from-model"></a>
### 5.5 Infer Missing Brands

Leverage the cleaned model names to backfill missing `Brand` entries using a curated brand-model lookup table.


When a model is known but `Brand` is missing, the helper below fills the gap using `mapping/brand_model_mapping.json` while logging before/after statistics.


This step relies on a maintained `brand_model_mapping.json` file; extend the mapping whenever new model names appear in fresh data pulls.


In [14]:
def fill_brand_from_model(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/brand_model_mapping.json"
) -> pd.DataFrame:
    """
    Fills df['Brand'] when:
      - Brand is NaN
      - model has a value
      - model exists in the JSON (brand_model_mapping.json)
    """
    with open(mapping_path, "r", encoding="utf-8") as f:
        model_to_brand = json.load(f)

    before = df["Brand"].isna().sum()
    print(f"[DEBUG] Brand NaN before: {before}")

    mask = df["Brand"].isna() & df["model"].notna()
    mapped = df.loc[mask, "model"].map(model_to_brand)

    fill_mask = mask.copy()
    fill_mask.loc[mask] = mapped.notna()

    df.loc[fill_mask, "Brand"] = mapped[mapped.notna()]

    after = df["Brand"].isna().sum()
    print(f"[DEBUG] Brand NaN after: {after}")
    print("[DEBUG] Filled examples (model -> Brand):")
    print(
        df.loc[fill_mask, ["model", "Brand"]]
          .head(4)
          .to_string(index=False)
    )

    return df

# Apply brand backfilling to both splits
train = fill_brand_from_model(train, "../mapping/brand_model_mapping.json")
test = fill_brand_from_model(test, "../mapping/brand_model_mapping.json")


[DEBUG] Brand NaN before: 1521
[DEBUG] Brand NaN after: 116
[DEBUG] Filled examples (model -> Brand):
model      Brand
T-Roc Volkswagen
   A3       Audi
  i20    Hyundai
 Polo Volkswagen
[DEBUG] Brand NaN before: 649
[DEBUG] Brand NaN after: 42
[DEBUG] Filled examples (model -> Brand):
  model         Brand
Mokka X          Opel
A-Class Mercedes-Benz
 Tucson       Hyundai
    i20       Hyundai


<a id="sec-outliers"></a>
## 6. Outlier & Anomaly Handling

Address domain violations identified during exploration so later modeling steps receive realistic inputs.


The following checks either remove implausible records (e.g., pre-2000 vehicles) or mark suspect values as `NaN` so they can be imputed responsibly in downstream notebooks.


<a id="sec-outliers-year"></a>
### 6.1 Year Before 2000


In [15]:
# Inspect year values prior to 1990 as a sanity check before filtering
print("Year values before 1990:")
print(train.loc[train['year'] < 1990, 'year'].unique())


Year values before 1990:
<IntegerArray>
[1970]
Length: 1, dtype: Int64


Only a handful of listings fall before the 2000 cutoff used in the competition scope, so we remove them from the training data.


In [16]:
# Drop training records with model year before 2000 (competition scope)
train = train[train['year'] >= 2000]


<a id="sec-outliers-engine"></a>
### 6.2 Engine Size Limits


Based on exploration and domain knowledge we expect engines between roughly 0.8L and 6.2L; anything outside that range is marked as missing.


In [17]:
# Flag engine sizes outside the accepted range and set them to NaN
low_engine_size_count = train[train['engineSize'] < 1.0].shape[0]
high_engine_size_count = train[train['engineSize'] > 6.2].shape[0]
print(f"Number of entries with engine size below 1.0L:  {low_engine_size_count}")
print(f"Number of entries with engine size above 6.2L:  {high_engine_size_count}")

train.loc[((train['engineSize'] > 6.2) | (train['engineSize'] < 0.8)), 'engineSize'] = np.nan
test.loc[((test['engineSize'] > 6.2) | (test['engineSize'] < 0.8)), 'engineSize'] = np.nan


Number of entries with engine size below 1.0L:  562
Number of entries with engine size above 6.2L:  1


<a id="sec-outliers-rounding"></a>
### 6.3 Year Rounding Errors


Listings with fractional model years are artefacts of prior processing; we reset them to `NaN` to avoid introducing impossible timestamps. The additional guard keeps placeholder negative years from affecting mileage entries.


In [18]:
# Remove fractional model years and guard against negative placeholders
train.loc[train['year'] % 1 != 0, 'year'] = np.nan
test.loc[test['year'] % 1 != 0, 'year'] = np.nan

# Legacy guard: clear mileage entries tied to negative year placeholders
train.loc[train['year'] < 0, 'mileage'] = np.nan
test.loc[test['year'] < 0, 'mileage'] = np.nan


<a id="sec-outliers-tax"></a>
### 6.4 Negative Tax Values


In [19]:
# Negative road-tax entries are invalid, so mark them as missing for later imputation
train.loc[train['tax'] < 0, 'tax'] = np.nan
test.loc[test['tax'] < 0, 'tax'] = np.nan


<a id="sec-outliers-mileage"></a>
### 6.5 Negative Mileage Values


In [20]:
# Mileage cannot be negative; treat these rows as missing mileage information
train.loc[train['mileage'] < 0, 'mileage'] = np.nan
test.loc[test['mileage'] < 0, 'mileage'] = np.nan


<a id="sec-outliers-mpg"></a>
### 6.6 MPG vs Fuel Type


In [21]:
# Clip unrealistic mpg values based on fuel-type heuristics derived from exploration
unusual_mpg_mask = ((train['mpg'] > 150) & (train["fuelType"] == "Electric")) |        ((train['mpg'] < 70) & (train["fuelType"] == "Electric")) |        ((train['mpg'] > 100) & (train["fuelType"] == "Hybrid")) |        ((train['mpg'] < 35) & (train["fuelType"] == "Hybrid")) |        ((train['mpg'] > 80) & (train["fuelType"] != "Hybrid") & (train["fuelType"] != "Electric")) |        ((train['mpg'] < 8) & (train["fuelType"] != "Hybrid")& (train["fuelType"] != "Electric"))
train.loc[unusual_mpg_mask, 'mpg'] = np.nan

unusual_mpg_mask_test = ((test['mpg'] > 150) & (test["fuelType"] == "Electric")) |        ((test['mpg'] < 70) & (test["fuelType"] == "Electric")) |        ((test['mpg'] > 100) & (test["fuelType"] == "Hybrid")) |        ((test['mpg'] < 35) & (test["fuelType"] == "Hybrid")) |        ((test['mpg'] > 80) & (test["fuelType"] != "Hybrid") & (test["fuelType"] != "Electric")) |        ((test['mpg'] < 8) & (test["fuelType"] != "Hybrid")& (test["fuelType"] != "Electric"))
test.loc[unusual_mpg_mask_test, 'mpg'] = np.nan


<a id="sec-outliers-owners"></a>
### 6.7 Previous Owners


In [22]:
# Negative owner counts break business logic; treat them as missing values
train.loc[train['previousOwners'] < 0, 'previousOwners'] = np.nan
test.loc[test['previousOwners'] < 0, 'previousOwners'] = np.nan


<a id="sec-outliers-future-year"></a>
### 6.8 Future Model Years


The dataset snapshot represents 2020; any listing newer than that year likely stems from data entry errors and is marked as missing.


In [23]:
train.loc[train['year'] > 2020, 'year'] = np.nan
test.loc[test['year'] > 2020, 'year'] = np.nan


<a id="sec-outliers-paint"></a>
### 6.9 Paint Quality Range


Paint quality should lie between 0% and 100%; any negative or over-cap values are removed for later treatment.


In [24]:
train.loc[(train['paintQuality%'] > 100) | (train['paintQuality%'] < 0), 'paintQuality%'] = np.nan
test.loc[(test['paintQuality%'] > 100) | (test['paintQuality%'] < 0), 'paintQuality%'] = np.nan


<a id="sec-outliers-unknown-transmission"></a>
### 6.10 Unknown Transmission


Entries labelled "Unknown" for transmission offer no signal for modelling, so we treat them as missing to allow later imputation.


In [25]:
train.loc[train['transmission'] == 'Unknown', 'transmission'] = np.nan
test.loc[test['transmission'] == 'Unknown', 'transmission'] = np.nan


<a id="sec-export"></a>
## 7. Export Processed Data

Persist the cleaned datasets so subsequent notebooks can consume a consistent starting point.


In [26]:
# Save the processed train/test dataframes for downstream modelling
PROCESSED_CSV = os.path.join(data_dir, "processed_data/11_processed_train_data.csv")
print("Saving processed file to:", PROCESSED_CSV)
train.to_csv(PROCESSED_CSV, index=False)

PROCESSED_CSV_TEST = os.path.join(data_dir, "processed_data/11_processed_test_data.csv")
print("Saving processed file to:", PROCESSED_CSV_TEST)
test.to_csv(PROCESSED_CSV_TEST, index=False)


Saving processed file to: ../data/processed_data/11_processed_train_data.csv
Saving processed file to: ../data/processed_data/11_processed_test_data.csv


<a id="sec-appendix"></a>
## 8. Appendix: Missing Value Summary

Compile post-cleaning missing-value counts for both splits to inform downstream imputation strategies.


In [27]:
# Remaining missing-value counts for the cleaned training data
nan_summary = train.isna().sum().reset_index()
nan_summary.columns = ['Column', 'NaN Count']
nan_summary = nan_summary[nan_summary['NaN Count'] > 0]
nan_summary = nan_summary.sort_values(by='NaN Count', ascending=False)
nan_summary


,Column,NaN Count
9,mpg,9004
8,tax,8108
5,transmission,2218
10,engineSize,2055
12,previousOwners,1888
11,paintQuality%,1853
6,mileage,1801
2,model,1653
13,hasDamage,1521
7,fuelType,1479


In [28]:
# Remaining missing-value counts for the cleaned test data
nan_summary_test = test.isna().sum().reset_index()
nan_summary_test.columns = ['Column', 'NaN Count']
nan_summary_test = nan_summary_test[nan_summary_test['NaN Count'] > 0]
nan_summary_test = nan_summary_test.sort_values(by='NaN Count', ascending=False)
nan_summary_test


,Column,NaN Count
8,mpg,3840
7,tax,3469
3,year,1007
4,transmission,968
9,engineSize,875
5,mileage,859
10,paintQuality%,793
11,previousOwners,765
2,model,734
6,fuelType,656


[Back to top](#top)
